In [ ]:
# Setup, Version check and Common imports

# Python ≥ 3.7 is required
import sys
assert sys.version_info >= (3, 7)

# TensorFlow ≥ 2.8 is required
import tensorflow as tf
from packaging import version

assert version.parse(tf.__version__) >= version.parse("2.8.0")

# Common imports
import numpy as np
import os
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

import tensorflow_hub as hub

from matplotlib import gridspec

print('Python version: ', sys.version_info)
print('TF version: ', tf.__version__)
print('Keras version: ', keras.__version__)
print('GPU is', 'available' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE')

In [ ]:
#Define image loading and visualization functions

import functools
from PIL import Image

def crop_center(image):
  """Returns a cropped square image."""
  shape = image.shape
  new_shape = min(shape[1], shape[2])
  offset_y = max(shape[1] - shape[2], 0) // 2
  offset_x = max(shape[2] - shape[1], 0) // 2
  image = tf.image.crop_to_bounding_box(
      image, offset_y, offset_x, new_shape, new_shape)
  return image

def load_image_from_path(path, image_size):
    img = Image.open(path)
    img = img.resize(image_size)
    img = np.array(img) / 255.0
    img = img.astype(np.float32)[np.newaxis, ...]
    return img

def show_n(images, titles=('',)):
  n = len(images)
  image_sizes = [image.shape[1] for image in images]
  w = (image_sizes[0] * 6) // 320
  plt.figure(figsize=(w * n, w))
  gs = gridspec.GridSpec(1, n, width_ratios=image_sizes)
  for i in range(n):
    plt.subplot(gs[i])
    plt.imshow(images[i][0], aspect='equal')
    plt.axis('off')
    plt.title(titles[i] if len(titles) > i else '')



output_image_size = 384

In [ ]:
# Upload file Images.zip to current directory

!unzip Images.zip

!rm Images.zip

In [ ]:
# Move to images folder

%cd images/

In [ ]:
# Load example images

# Original image (Content Image)
content_name = 'Golden_Gate.jpg'

#  Style Image
style_name = 'The_Great_Wave.jpg'


# The content image size can be arbitrary.
content_img_size = (output_image_size, output_image_size)

# The style prediction model was trained with image size 256 and it's the
# recommended image size for the style image (though, other sizes work as
# well but will lead to different results).
style_img_size = (256, 256)

content_image = load_image_from_path(content_name, content_img_size)
style_image = load_image_from_path(style_name, style_img_size)

style_image = tf.nn.avg_pool(style_image, ksize=[3,3], strides=[1,1], padding='SAME')
show_n([content_image, style_image], ['Content image', 'Style image'])

In [ ]:
# Load TF Hub module.
# Retrieve a pretrained style transfer model
# https://arxiv.org/pdf/1705.06830.pdf

hub_handle = 'https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2'
hub_module = hub.load(hub_handle)


In [ ]:
# Model interface
# content_image, style_image, and stylized_image are 4-D tensors
# Shape [batch_size, image_height, image_width, 3].

# In the following examples, batch size is always 1 (isolated images)

# Apply style to selected image

outputs = hub_module(tf.constant(content_image), tf.constant(style_image))
stylized_image = outputs[0]

In [ ]:
# Visualize 3 images

show_n([content_image, style_image, stylized_image], titles=['Original content image', 'Style image', 'Stylized image'])

In [ ]:
# Apply different styles to several content images

# Original image (Content Image)
content_name = 'Sea_Turtle.jpg'  # @param ['Golden_Gate.jpg', 'Sea_Turtle.jpg', 'Tuebingen.jpg']

#  Style Image
style_name = 'The_Great_Wave.jpg'  # @param ['The_Great_Wave.jpg', 'Composition_7.jpg','Demoiselles_dAvignon.jpg', 'Nantes.jpg', 'Starry_Night.jpg', 'Still_Life.jpg', 'The_Scream.jpg', 'Violin.jpg']

content_image = load_image_from_path(content_name, content_img_size)
style_image = load_image_from_path(style_name, style_img_size)

outputs = hub_module(tf.constant(content_image), tf.constant(style_image))
stylized_image = outputs[0]
show_n([content_image, style_image, stylized_image], titles=['Original content image', 'Style image', 'Stylized image'])


**Try with your own images**



In [ ]:
# Upload your own images to the images directory


image_size=(384,384)
image_name = '1.JPG' # @param {type:"string"}

image = tf.io.read_file(image_name)
input_i = tf.image.decode_image(image, channels=3, dtype=tf.float32)[tf.newaxis, ...]

input_i = crop_center(input_i)
input_i = tf.image.resize(input_i, image_size, preserve_aspect_ratio=True)

In [ ]:
input_i.shape

In [ ]:
# Select style

# The resulting image is saved to the current directory

style_name = 'The_Scream.jpg'  # @param ['The_Great_Wave.jpg', 'Composition_7.jpg','Demoiselles_dAvignon.jpg', 'Nantes.jpg', 'Starry_Night.jpg', 'Still_Life.jpg', 'The_Scream.jpg', 'Violin.jpg']

style_image = load_image_from_path(style_name, style_img_size)

outputs = hub_module(tf.constant(input_i), tf.constant(style_image))
stylized_image = outputs[0]
show_n([input_i, style_image, stylized_image], titles=['Original content image', 'Style image', 'Stylized image'])

save_file = 'temp.png' # @param {type:"string"}
plt.savefig(save_file)